# HyperKT Objective 01 — Stage 9F: Event-Level Sequence Audit & Conditional Protocol Lock

**Purpose:** verify that Stage 2 train/validation target events map exactly once to the chronological Stage 1 interaction records, preserve the original Stage 9E results, and produce a *conditional* protocol decision. **No model training or test-set predictive evaluation occurs here.**

**Inputs:** `hyperkt_stage1.zip`, `hyperkt_stage2(2).zip`, `hyperkt_stage9e.zip`. Optional: Stage 9E checkpoint folder ZIP. **Runtime: CPU.**

In [ ]:
!pip -q install pyarrow pandas numpy scipy

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, json, hashlib, numpy as np, pandas as pd, scipy.stats as stats
from collections import defaultdict
from datetime import datetime, timezone
uploads=files.upload()
required=['hyperkt_stage1.zip','hyperkt_stage2','hyperkt_stage9e.zip']
assert any('stage1' in x.lower() and x.endswith('.zip') for x in uploads)
assert any('stage2' in x.lower() and x.endswith('.zip') for x in uploads)
assert any('stage9e' in x.lower() and x.endswith('.zip') for x in uploads)
def locate(t):return next(k for k in uploads if t in k.lower() and k.lower().endswith('.zip'))
z1=zipfile.ZipFile(locate('stage1'));z2=zipfile.ZipFile(locate('stage2'));z9=zipfile.ZipFile(locate('stage9e'))
print('Archives:',len(z1.namelist()),len(z2.namelist()),len(z9.namelist()))

In [ ]:
vocab=json.loads(z2.read('skill_vocab.json'))
manifest1=json.loads(z1.read('manifest.json'));manifest2=json.loads(z2.read('manifest.json'))
assert manifest1['source_sha256']==manifest2['source_sha256'], 'Source dataset mismatch'
assert json.loads(z1.read('skill_vocab.json'))==vocab, 'Skill vocabulary mismatch'
def load_events(split):
    d=pd.read_parquet(io.BytesIO(z1.read(f'{split}_interactions.parquet')))
    print(split,'columns:',list(d.columns),'rows:',len(d))
    for k in ['user','skill','correct','order']:
        assert k in d.columns, f'{split}: missing {k}'
    d=d.copy();d['user']=d['user'].astype(str);d['skill_id']=d['skill'].astype(str).map(vocab)
    assert d['skill_id'].notna().all(),f'{split}: unknown skill'
    # stable ordering matches the sequence-builder's chronological ordering only if it used order ascending
    d=d.sort_values(['user','order'],kind='mergesort')
    return d
def load_sequences(split):
    a=np.load(io.BytesIO(z2.read(f'{split}_sequences.npz')),allow_pickle=False)
    return {k:a[k] for k in a.files}
train_events=load_events('train');val_events=load_events('val')
train_seq=load_sequences('train');val_seq=load_sequences('val')

## Event-level reconstruction
For each student, reconstruct the expected target stream from the original ordered interactions. The first interaction is context; every subsequent interaction must appear exactly once as a target, in order. Each window's first history token must be the immediately preceding event. This detects overlapping or skipped target events **if Stage 1 and Stage 2 use the same event ordering**. We also check adjacent history/target shifts within each window.

In [ ]:
def audit(split,events,seq):
    groups={u:g for u,g in events.groupby('user',sort=False)}
    windows=defaultdict(list)
    for i,u in enumerate(seq['user_id'].astype(str)):
        windows[u].append(i)
    issues=[]; stats_rows=[];total_targets=0
    for user,g in groups.items():
        s=g['skill_id'].to_numpy(dtype=np.int64)
        c=g['correct'].to_numpy(dtype=np.int64)
        idxs=windows.get(user,[])
        cursor=1
        for wi in idxs:
            mask=seq['mask'][wi].astype(bool)
            if not np.array_equal(mask,np.arange(len(mask))<mask.sum()):
                issues.append((split,user,wi,'non-prefix mask'))
            t=int(mask.sum())
            if not t:issues.append((split,user,wi,'empty target window'));continue
            hs=seq['history_skill'][wi,mask];hc=seq['history_correct'][wi,mask]
            ts=seq['target_skill'][wi,mask];tc=seq['target_correct'][wi,mask]
            if cursor+t>len(s):issues.append((split,user,wi,'target overflow'));continue
            if not (np.array_equal(ts,s[cursor:cursor+t]) and np.array_equal(tc,c[cursor:cursor+t])):
                issues.append((split,user,wi,'target stream differs from Stage1'))
            if not (np.array_equal(hs,s[cursor-1:cursor+t-1]) and np.array_equal(hc,c[cursor-1:cursor+t-1])):
                issues.append((split,user,wi,'history stream differs from Stage1'))
            cursor+=t;total_targets+=t
        expected=max(0,len(s)-1)
        if cursor-1!=expected:issues.append((split,user,-1,f'coverage {cursor-1}/{expected}'))
        stats_rows.append({'split':split,'user':user,'events':len(s),'targets_expected':expected,'targets_recovered':cursor-1,'windows':len(idxs)})
    unexpected=set(windows)-set(groups)
    for u in unexpected:issues.append((split,u,-1,'user not in Stage1'))
    return {'split':split,'event_rows':len(events),'students':len(groups),'windows':len(seq['user_id']),
            'targets':total_targets,'expected_targets':sum(max(0,len(g)-1) for g in groups.values()),
            'issue_count':len(issues),'issues_sample':issues[:100]},pd.DataFrame(stats_rows),issues
rtr,ctr,itr=audit('train',train_events,train_seq)
rva,cva,iva=audit('val',val_events,val_seq)
print(json.dumps([rtr,rva],indent=2))
print('Student overlap:',len(set(ctr.user)&set(cva.user)))

## Model selection evidence and checkpoint availability
Stage 9E is exploratory. A protocol lock is **not** the same as evidence of superiority. The original ungated HyperKT is retained as the proposed architecture because the gated variant did not demonstrate a gain. DKT is the strongest validation baseline. Stage 9E ZIP contains metrics, not necessarily weights; confirm separately that the exact selected checkpoints exist.

In [ ]:
results=pd.read_csv(io.BytesIO(z9.read('all_results.csv')))
summary=pd.read_csv(io.BytesIO(z9.read('summary.csv')))
print(summary.to_string(index=False))
print('Stage9E ZIP checkpoint entries:',[x for x in z9.namelist() if x.endswith(('.pt','.pth','.ckpt'))][:5])
print('Columns:',list(results.columns))
print('Uploaded checkpoint ZIPs:',[k for k in uploads if 'checkpoint' in k.lower() or 'stage9e' in k.lower() and k!='hyperkt_stage9e.zip'])

In [ ]:
import os
out=Path('/content/hyperkt_stage9f');out.mkdir(exist_ok=True)
student_overlap=len(set(ctr.user)&set(cva.user))
event_pass=(rtr['issue_count']==0 and rva['issue_count']==0 and student_overlap==0)
# Do not assume recovered checkpoints from a metrics-only archive
checkpoint_archives=[k for k in uploads if k!=locate('stage9e') and k!=locate('stage1') and k!=locate('stage2') and k.endswith('.zip')]
checkpoint_count=0
for fn in checkpoint_archives:
    with zipfile.ZipFile(fn) as z:
        checkpoint_count+=sum(x.endswith(('.pt','.pth','.ckpt')) for x in z.namelist())
checkpoint_pass=checkpoint_count>=10 # at least five DKT + five original HyperKT; exact names must still be verified
report={'stage':'9F','timestamp_utc':datetime.now(timezone.utc).isoformat(),
        'stage1_sha256':hashlib.sha256(open(locate('stage1'),'rb').read()).hexdigest(),
        'stage2_sha256':hashlib.sha256(open(locate('stage2'),'rb').read()).hexdigest(),
        'stage9e_sha256':hashlib.sha256(open(locate('stage9e'),'rb').read()).hexdigest(),
        'train_audit':rtr,'validation_audit':rva,'student_overlap':student_overlap,
        'event_uniqueness_pass':bool(event_pass),'checkpoint_files_in_optional_archives':checkpoint_count,
        'checkpoint_inventory_preliminary_pass':bool(checkpoint_pass),
        'baseline_fidelity':'Independent implementations; exact published reproduction not verified',
        'test_predictions_evaluated':False,
        'recommended_proposed_model':'Original ungated HyperKT (Stage9E matched)',
        'recommended_comparator':'DKT (Stage9E matched)',
        'decision':('CONDITIONAL_LOCK_PENDING_CHECKPOINT_AND_FIDELITY' if event_pass else 'BLOCKED_EVENT_ALIGNMENT_FAILURE')}
(out/'event_audit.json').write_text(json.dumps(report,indent=2))
pd.concat([ctr,cva],ignore_index=True).to_csv(out/'student_coverage.csv',index=False)
summary.to_csv(out/'stage9e_summary_copy.csv',index=False)
(out/'audit_issues.json').write_text(json.dumps({'train':itr[:500],'validation':iva[:500]},indent=2))
print('DECISION:',report['decision'])
print('Checkpoint files found:',checkpoint_count)
if not event_pass:print('STOP: event-level audit did not pass. Do not evaluate test.')

## Protocol-lock checklist

- **Data integrity:** Stage 1/2 source and vocabulary hashes match; train/validation events align exactly and each target appears once (only if the audit reports zero issues).
- **Model selection:** retain original ungated HyperKT and matched DKT; do not select a model based on test performance.
- **Training:** five seeds `[42, 123, 2024, 3407, 7777]`; Stage 9E settings: batch 32, AdamW LR 2e-4, 20 epochs maximum, patience 5, hidden 128, dropout 0.1.
- **Checkpoints:** recover exact Stage 9E `best.pt` files from Drive, verify hashes and strict loading before test inference. A ZIP containing metrics alone is insufficient.
- **Baselines:** label AKT-inspired and other independent implementations accurately; do not imply official architectural fidelity.
- **Test evaluation:** only after final protocol sign-off, once, using frozen checkpoints and predeclared metrics. Never use test feedback for redesign.
- **Inference statistics:** five seeds are repeated training runs on one cohort, not five independent student samples. Report seed variability separately from student-level uncertainty.

**Decision is conditional** until all applicable checks are complete.

In [ ]:
import shutil
archive=shutil.make_archive('/content/hyperkt_stage9f','zip',root_dir='/content',base_dir='hyperkt_stage9f')
print('Created:',archive)
files.download(archive)